In [4]:
import pandas as pd
import numpy as np
import time
import json

# ==========================================
# LAYER 1: MULTI-SOURCE DATA INGESTION & COUPLING
# ==========================================
def generate_enterprise_data(records=5000):
    """Simulates real-world fragmented data from two distinct legacy databases."""
    np.random.seed(101)

    # Dataset A: Core Application Database (User ID and declared income)
    app_db = pd.DataFrame({
        'application_id': [f"APP-{1000+i}" for i in range(records)],
        'declared_income': np.random.choice([np.random.randint(20000, 160000), np.nan, -25000], size=records, p=[0.94, 0.04, 0.02])
    })

    # Dataset B: Credit Bureau & Risk Database (User ID, bureau score, and verified tax income)
    bureau_db = pd.DataFrame({
        'application_id': [f"APP-{1000+i}" for i in range(records)],
        'credit_score': np.random.randint(300, 850, size=records),
        'verified_tax_income': np.random.randint(15000, 150000, size=records)
    })

    # Introduce data integration failures (Missing records in Bureau DB)
    bureau_db = bureau_db.drop(bureau_db.sample(frac=0.03).index)

    return app_db, bureau_db

# ==========================================
# LAYER 2: DATA INTEGRATION & QUALITY ASSURANCE (ETL)
# ==========================================
def clean_and_integrate_pipeline(app_df, bureau_df):
    """Performs an advanced relational join and handles multi-source discrepancies."""
    # Simulating a SQL Left Join to preserve all incoming core applications
    merged_df = pd.merge(app_df, bureau_df, on='application_id', how='left')

    # FIX: Explicitly cast NumPy int64 types to native Python ints for JSON serialization
    metrics = {
        'initial_ingestion_count': int(len(app_df)),
        'missing_bureau_records': int(merged_df['credit_score'].isna().sum()),
        'missing_declared_income': int(merged_df['declared_income'].isna().sum())
    }

    # Data Scrubbing & Imputation Rules
    merged_df['declared_income'] = merged_df['declared_income'].fillna(0)
    merged_df['credit_score'] = merged_df['credit_score'].fillna(500) # Subprime default fallback
    merged_df['verified_tax_income'] = merged_df['verified_tax_income'].fillna(0)

    return merged_df, metrics

# ==========================================
# LAYER 3: MULTI-STAGE REAL-TIME ROUTING ENGINE
# ==========================================
def execute_risk_decision_router(row):
    """
    Advanced routing logic simulating real-time orchestration.
    Evaluates income variance, fraud discrepancies, and risk criteria.
    """
    start_time = time.perf_counter()

    # Stage 3a: Exception Handling / Data Integrity Flags
    if row['declared_income'] <= 0 or row['credit_score'] < 300:
        execution_time = (time.perf_counter() - start_time) * 1000 + np.random.uniform(2, 5)
        return "IMMEDIATE_REJECT", execution_time, "Data Integrity / Fraud Anomaly"

    # Stage 3b: Income Variance & Discrepancy Verification Checking
    income_variance = abs(row['declared_income'] - row['verified_tax_income']) / max(row['verified_tax_income'], 1)

    # Stage 3c: Orchestration Routing Gates
    if row['credit_score'] >= 750 and income_variance < 0.15:
        # High score, low discrepancy -> Straight-Through Processing (STP)
        execution_time = (time.perf_counter() - start_time) * 1000 + np.random.uniform(10, 25)
        return "STRAIGHT_THROUGH_APPROVAL", execution_time, "Optimal Metrics Match"

    elif row['credit_score'] >= 600 and income_variance <= 0.40:
        # Mid-tier risk profile -> Secondary Verification Required
        execution_time = (time.perf_counter() - start_time) * 1000 + np.random.uniform(120, 280)
        return "MANUAL_VERIFICATION_REQUIRED", execution_time, "High Income Discrepancy Gate"

    else:
        # High Risk Profile -> Bureau Escalation Path
        execution_time = (time.perf_counter() - start_time) * 1000 + np.random.uniform(400, 650)
        return "BUREAU_ESCALATION_REJECT", execution_time, "Subprime Portfolio Threshold Breach"

# ==========================================
# LAYER 4: EXECUTION & METRIC GENERATION ENGINE
# ==========================================
# 1. Run Data Ingestion
apps, bureau = generate_enterprise_data(records=5000)

# 2. Run ETL / Data Quality Layer
integrated_data, pipeline_metrics = clean_and_integrate_pipeline(apps, bureau)

# 3. Process records through the Routing Engine
routing_results = integrated_data.apply(execute_risk_decision_router, axis=1)
integrated_data['decision_outcome'], integrated_data['latency_ms'], integrated_data['routing_reason'] = zip(*routing_results)

# 4. Generate Strategic Strategy Insights Table
performance_summary = integrated_data.groupby('decision_outcome').agg(
    Volume_Count=('latency_ms', 'count'),
    Average_Latency_ms=('latency_ms', 'mean'),
    P95_Worst_Case_Latency_ms=('latency_ms', lambda x: np.percentile(x, 95))
).reset_index()

# Print out Executive Summary for Resume Verification
print("=== ENTERPRISE RISK PIPELINE ARCHITECTURE REPORT ===")
print(json.dumps(pipeline_metrics, indent=4))
print("\n=== ROUTING PERFORMANCE LOGIC SUMMARY ===")
print(performance_summary.to_string(index=False))

=== ENTERPRISE RISK PIPELINE ARCHITECTURE REPORT ===
{
    "initial_ingestion_count": 5000,
    "missing_bureau_records": 150,
    "missing_declared_income": 217
}

=== ROUTING PERFORMANCE LOGIC SUMMARY ===
            decision_outcome  Volume_Count  Average_Latency_ms  P95_Worst_Case_Latency_ms
    BUREAU_ESCALATION_REJECT          4144          524.956989                 637.877582
            IMMEDIATE_REJECT           326            3.538225                   4.893150
MANUAL_VERIFICATION_REQUIRED           432          200.975668                 271.491566
   STRAIGHT_THROUGH_APPROVAL            98           17.966509                  24.114932
